# Cómo procesa MobileNetV2 una hoja de mango

Seguimos el bloque real `features[3]` de la arquitectura utilizada en las dos corridas. Una entrada RGB de 224 × 224 llega al bloque como 24 mapas de 56 × 56.

In [1]:
import torch
from torch import nn
from torchvision.models import MobileNet_V2_Weights, mobilenet_v2

torch.set_num_threads(2)
model = mobilenet_v2(weights=MobileNet_V2_Weights.IMAGENET1K_V1).eval()
print("Cabeza original:", model.classifier)
model.classifier[1] = nn.Linear(model.last_channel, 8)
block = model.features[3]
print(block)
with torch.no_grad():
    image = torch.zeros(1, 3, 224, 224)
    x = model.features[:3](image)
    z = x
    print("Entrada:", tuple(x.shape))
    stages = ["Expansión", "Depthwise", "Proyección", "BatchNorm"]
    for name, layer in zip(stages, block.conv, strict=True):
        z = layer(z)
        print(name, tuple(z.shape))
    print("Suma residual:", block.use_res_connect)
    torch.testing.assert_close(block(x), x + z)

Cabeza original: Sequential(
  (0): Dropout(p=0.2, inplace=False)
  (1): Linear(in_features=1280, out_features=1000, bias=True)
)
InvertedResidual(
  (conv): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(24, 144, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (1): BatchNorm2d(144, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU6(inplace=True)
    )
    (1): Conv2dNormActivation(
      (0): Conv2d(144, 144, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=144, bias=False)
      (1): BatchNorm2d(144, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU6(inplace=True)
    )
    (2): Conv2d(144, 24, kernel_size=(1, 1), stride=(1, 1), bias=False)
    (3): BatchNorm2d(24, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  )
)
Entrada: (1, 24, 56, 56)
Expansión (1, 144, 56, 56)
Depthwise (1, 144, 56, 56)
Proyección (1, 24, 56, 56)
BatchNorm (1, 24, 56, 56)
Suma residual: True


La **expansión** usa una convolución pointwise de 1 × 1 para mezclar los 24 canales y producir 144 canales, seguida de normalización y ReLU6. La **depthwise** usa 144 grupos. Cada filtro de 3 × 3 recorre un solo canal. El stride 1 y el relleno conservan los 56 × 56 píxeles.

La **proyección** pointwise vuelve a mezclar los canales y comprime 144 a 24. La convolución y la normalización finales forman el **cuello de botella lineal**. Aplicar ReLU en esta representación estrecha descartaría componentes negativas y podría perder información. El bloque se llama *residual invertido* porque filtra en la representación expandida y conecta los extremos estrechos.

En este bloque, el stride es 1 y los canales de entrada y salida coinciden. Por eso se puede sumar la entrada a la proyección elemento a elemento. En el bloque anterior, el stride 2 reduce la resolución y cambia el número de canales.

In [2]:
with torch.no_grad():
    previous = model.features[:2](image)
    reduced = model.features[2](previous)
    print(tuple(previous.shape), "→", tuple(reduced.shape))
    print("Suma residual:", model.features[2].use_res_connect)
    print("Logits de la cabeza adaptada:", tuple(model(image).shape))

(1, 16, 112, 112) → (1, 24, 56, 56)
Suma residual: False
Logits de la cabeza adaptada: (1, 8)
